In [60]:
import pandas as pd

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [61]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [62]:
train_df = train_df.drop("Cabin", axis= 1 )

In [63]:
#Data Cleaning
from sklearn.impute import SimpleImputer
#Numerical Imputer
num_imputer = SimpleImputer(strategy="median")
train_df[["Age"]] = num_imputer.fit_transform(train_df[["Age"]])
#Categorial Imputer
cat_imputer = SimpleImputer(strategy="most_frequent")
train_df[["Embarked"]] = cat_imputer.fit_transform(train_df[["Embarked"]])

In [64]:
train_df = train_df.drop(["PassengerId" , "Name" , "Ticket"] , axis = 1 , inplace= False)

In [65]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       891 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Embarked  891 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 55.8+ KB


In [66]:
#Data Encoding
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
encoder_1 = LabelEncoder()
train_df["Sex"] = encoder.fit_transform(train_df["Sex"])
train_df["Embarked"] = encoder_1.fit_transform(train_df["Embarked"])

In [67]:
#Feature Enginering
train_df["Family_Size"] = train_df["SibSp"] + train_df["Parch"] + 1 
test_df["Family_Size"] = test_df["SibSp"] + test_df["Parch"] + 1

train_df["isAlone"] = (train_df["Family_Size"] == 1).astype(int)
test_df["isAlone"] = (test_df["Family_Size"] == 1).astype(int)

In [86]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Survived     891 non-null    int64  
 1   Pclass       891 non-null    int64  
 2   Sex          891 non-null    int32  
 3   Age          891 non-null    float64
 4   SibSp        891 non-null    int64  
 5   Parch        891 non-null    int64  
 6   Fare         891 non-null    float64
 7   Embarked     891 non-null    int32  
 8   Family_Size  891 non-null    int64  
 9   isAlone      891 non-null    int32  
dtypes: float64(2), int32(3), int64(5)
memory usage: 59.3 KB


In [87]:
x = train_df.drop("Survived" , axis= 1)
y = train_df["Survived"]

In [88]:
#Train-Test split
from sklearn.model_selection import train_test_split

x_train , x_test , y_train , y_test = train_test_split(x , y , test_size= 0.2 , random_state=42)


In [89]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(n_estimators= 100 , random_state= 42)
rf.fit(x_train , y_train)

RandomForestClassifier(random_state=42)

In [90]:
y_pred = rf.predict(x_test)
accuracy_score(y_test , y_pred)

0.8156424581005587

In [91]:
from sklearn.model_selection import cross_val_score

value = cross_val_score(rf , x_train , y_train , cv= 3 , scoring="accuracy")
print(f"The accuracy is {value.mean() * 100}")

The accuracy is 79.07255729295937


In [92]:
rf.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

In [93]:
#Fine Tuning the model

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

In [94]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(rf , param_grid , cv= 5 , n_jobs= -1 , scoring= "accuracy")
grid_search.fit(x_train , y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [5, 10, 15, None],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             scoring='accuracy')

In [95]:
y_pred = grid_search.predict(x_test)
accuracy_score(y_test , y_pred)

0.8100558659217877

In [96]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Pclass       418 non-null    int64  
 1   Sex          418 non-null    int32  
 2   Age          418 non-null    float64
 3   SibSp        418 non-null    int64  
 4   Parch        418 non-null    int64  
 5   Fare         418 non-null    float64
 6   Embarked     418 non-null    int32  
 7   Family_Size  418 non-null    int64  
 8   isAlone      418 non-null    int32  
dtypes: float64(2), int32(3), int64(4)
memory usage: 24.6 KB


In [97]:
test_df.drop(["PassengerId" ,"Name" , "Cabin" , "Ticket"] , axis= 1 , inplace=True)
test_df.info()

KeyError: "['PassengerId', 'Name', 'Cabin', 'Ticket'] not found in axis"

In [ ]:
#Impute
test_df[["Age"]] = num_imputer.transform(test_df[["Age"]])
test_df[["Embarked"]] = cat_imputer.transform(test_df[["Embarked"]])


In [ ]:
fare_imputer = SimpleImputer()
fare_imputer.fit(train_df[["Fare"]])
test_df[["Fare"]] = fare_imputer.transform(test_df[["Fare"]])

In [ ]:
#Data Encoding
test_df["Sex"] = encoder.transform(test_df["Sex"])
test_df["Embarked"] = encoder_1.transform(test_df["Embarked"])

In [ ]:
#Predictions
predictions = grid_search.best_estimator_.predict(test_df[x.columns])

test_orignal = pd.read_csv('test.csv')

submission = pd.DataFrame({
    'PassengerId' : test_orignal['PassengerId'],  
    'Survived' : predictions
})

submission.to_csv('submission.csv', index=False)

submission.to_csv('submission.csv' , index= False)

In [ ]:
# Make sure new features are included
print("Training features:", x.columns.tolist())
# Should include: ..., 'Family_Size', 'isAlone'

print("\nTest features:", test_df.columns.tolist())
# Should match training features

Training features: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Family_Size', 'isAlone']

Test features: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Family_Size', 'isAlone']
